# 使用 PyTorch 实现多头自注意力机制的逐步代码设计方案（中文说明版）

本 Notebook 结合题目描述，以逐步实现的方式展示如何在 **PyTorch** 中编写一个 `MultiHeadSelfAttention` 模块。

* **注释**：全部使用中文，详细说明每一步要做什么。
* **图表**：标题、坐标轴、图例等使用英文，避免字体问题。
* **图片保存**：所有生成的图片以 PNG 格式保存到 `outputs/images` 目录，并在 Notebook 中展示。
* **打印信息**：可中英混合，便于阅读。


In [ ]:
# 导入所需的基础库，并创建图片输出目录
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

# 中文注释：准备图片输出路径，确保目录存在，便于保存 PNG 图片
OUTPUT_DIR = Path("outputs") / "images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"图片将保存到: {OUTPUT_DIR.resolve()}")


## 1. 模块设计要点回顾

下面的实现严格按照题目给出的步骤展开：

1. 在 `__init__` 中检查形状合法性，定义生成 \(Q/K/V\) 的线性层与输出线性层，并可选加入 Dropout。
2. 在 `forward` 中完成：
   - 线性映射得到 \(Q/K/V\)；
   - 重塑并交换维度，将 `d_model` 拆成 `(num_heads, d_k)`；
   - 计算 scaled dot-product attention；
   - 拼接多头结果并映射回 `d_model` 维度。
3. 通过一个小例子验证输出形状，并可视化注意力权重热力图。


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    '''中文文档字符串：实现多头自注意力的核心模块。'''

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        # 中文注释：初始化时检查 d_model 能否被 num_heads 整除，确定每个头的维度 d_k
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 中文注释：定义生成 Q、K、V 的线性层，输出维度均为 d_model（即 num_heads * d_k）
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        # 中文注释：定义输出线性层，将多头拼接后的结果映射回 d_model 维度
        self.w_o = nn.Linear(d_model, d_model)

        # 中文注释：可选 Dropout，默认关闭，可通过参数调节
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, need_weights: bool = False):
        '''中文文档字符串：前向传播，支持返回注意力权重用于可视化。'''
        # 中文注释：x 形状应为 (batch_size, seq_len, d_model)
        batch_size, seq_len, _ = x.shape

        # 中文注释：通过线性层得到 Q、K、V，形状仍为 (B, L, d_model)
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        # 中文注释：重塑为多头形状 (B, L, num_heads, d_k)，再调整维度为 (B, num_heads, L, d_k)
        def reshape_to_heads(tensor):
            # 中文注释：使用 view/reshape + transpose 将 head 维提前，方便批量矩阵乘法
            return (
                tensor.view(batch_size, seq_len, self.num_heads, self.d_k)
                .permute(0, 2, 1, 3)
                .contiguous()
            )

        q = reshape_to_heads(q)
        k = reshape_to_heads(k)
        v = reshape_to_heads(v)

        # 中文注释：计算注意力得分 scores，使用缩放因子 sqrt(d_k) 避免梯度过小
        # scores 形状为 (B, num_heads, L, L)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)

        # 中文注释：对最后一维做 softmax 得到注意力权重，再可选应用 Dropout
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 中文注释：将注意力权重作用到 V 上，得到上下文向量 context，形状 (B, num_heads, L, d_k)
        context = torch.matmul(attn_weights, v)

        # 中文注释：把 head 维拼回去，先换维度到 (B, L, num_heads, d_k)，再合并 head 与 d_k
        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.view(batch_size, seq_len, self.d_model)

        # 中文注释：输出线性层得到最终结果，形状 (B, L, d_model)
        output = self.w_o(context)

        # 中文注释：按需返回注意力权重，便于外部可视化
        if need_weights:
            return output, attn_weights
        return output


## 2. 构造示例输入并验证输出形状

本节通过随机张量验证：
* 输出形状是否与输入一致；
* 是否支持返回注意力权重以便后续绘图。


In [ ]:
# 中文注释：设置随机种子，便于复现
torch.manual_seed(42)

# 中文注释：定义模型超参数
batch_size = 2
seq_len = 6
d_model = 16
num_heads = 4

# 中文注释：实例化多头自注意力层
mhsa = MultiHeadSelfAttention(d_model=d_model, num_heads=num_heads, dropout=0.1)

# 中文注释：创建随机输入张量，形状 (B, L, d_model)
x = torch.randn(batch_size, seq_len, d_model)

# 中文注释：执行前向传播，同时请求注意力权重方便后续绘图
with torch.no_grad():
    output, attn_weights = mhsa(x, need_weights=True)

print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"注意力权重形状: {attn_weights.shape}，期望 (B, num_heads, L, L)")


## 3. 可视化单个注意力头的权重热力图

选择第一个 batch、第一个注意力头，绘制权重矩阵并保存 PNG。

* 图像标题、坐标轴标签使用英文；
* 使用 seaborn/matplotlib 绘制；
* 同时展示并保存图片到 `outputs/images/attention_heatmap.png`。


In [ ]:
# 中文注释：选取第一个 batch、第一个头的注意力矩阵，形状 (L, L)
head0_matrix = attn_weights[0, 0].detach().cpu().numpy()

# 中文注释：准备保存路径
heatmap_path = OUTPUT_DIR / "attention_heatmap.png"

plt.figure(figsize=(6, 5))
sns.heatmap(head0_matrix, annot=True, fmt=".2f", cmap="Blues")
plt.title("Attention Weights (Head 1)")
plt.xlabel("Key Positions")
plt.ylabel("Query Positions")
plt.tight_layout()
plt.savefig(heatmap_path, dpi=150, format="png")
plt.show()

print(f"注意力热力图已保存: {heatmap_path}")


## 4. 小结与可扩展方向

* 该实现遵循标准 Transformer 的多头自注意力流程：线性映射、拆分多头、缩放点积注意力、拼接、输出映射。
* 可以在外部叠加残差连接与 LayerNorm 以组成完整的 Transformer Encoder Layer。
* 若在自回归场景需要掩码，可在 softmax 前对未来位置加上极大负数实现。
* 可进一步在 Notebook 中加入更多实验（如不同 head 数、序列长度），或将模块嵌入更大的模型进行验证。
